# VeriGym - Comparing abstraction policies

## Imports

In [ ]:
import stormpy
import gymnasium as gym
import numpy as np
from pprint import pprint
import logging

import verigym
from verigym.abstraction.gym_utils.transform_observation import ReplaceInfObservation
from verigym.frameworks.stormpy.stormpy_utils import build_stormpy_mdp
from verigym.policy.policy import RandomizedPolicy

logging.getLogger().setLevel(logging.ERROR)


def get_average_episode_length(trajectories):
    return np.mean([len(traj) for traj in trajectories])


def get_mean_reward_from_trajectories(trajectories):
    rewards = []
    for trajectory in trajectories:
        trajectory_rewards = list(map(lambda tup: tup[2], trajectory))
        rewards.append(np.sum(trajectory_rewards))
    return float(np.mean(rewards))

## 1. Environment

We use the  OpenAI gymnasium Cart Pole environment for this demo:

<img src="cart_pole.gif" width="400"/>


In [2]:
# 1. Load cart pole in gym
gym_env = gym.make("CartPole-v1")
print("Environment type: ", type(gym_env))

# 2. For discretization, replace (-inf, inf) observation bounds
gym_env = ReplaceInfObservation(gym_env, neg_inf=-5, pos_inf=5)
print("Observation space shape: ", gym_env.observation_space.shape)
print("Observation upper bounds: ", gym_env.observation_space.high)
print("Observation lower bounds: ", gym_env.observation_space.low)
print("Type of observation space: ", type(gym_env.observation_space))

Environment type:  <class 'gymnasium.wrappers.common.TimeLimit'>
Observation space shape:  (4,)
Observation upper bounds:  [4.8        5.         0.41887903 5.        ]
Observation lower bounds:  [-4.8        -5.         -0.41887903 -5.        ]
Type of observation space:  <class 'gymnasium.spaces.box.Box'>


Now, we load the `gym_env` into our framework, turning it into a (`gym`-like) `VeriGymEnv`:

In [3]:
# Create a VeriGymEnv from gym env
generative_model = verigym.GenerativeEnv.from_gymnasium(gym_env)

print("Environment type: ", type(generative_model))
print("Observation space shape: ", generative_model.observation_space.shape)
print("Observation upper bounds: ", generative_model.observation_space.high)
print("Observation lower bounds: ", generative_model.observation_space.low)
print("Type of observation space: ", type(generative_model.observation_space))

Environment type:  <class 'verigym.environments.generativeenv.GenerativeEnv_from_gym'>
Observation space shape:  (4,)
Observation upper bounds:  [4.8        5.         0.41887903 5.        ]
Observation lower bounds:  [-4.8        -5.         -0.41887903 -5.        ]
Type of observation space:  <class 'gymnasium.spaces.box.Box'>


The observation space is the same as before, but now we can apply learn the underlying model and perform any abstraction!

## 2. Abstraction

Now we: 
1. Apply a user-specified discretization to the state space
2. Learn the transition and reward functions through random search

In [4]:
abstracted_model = verigym.create_abstraction(
    original_env=generative_model,
    bin_edges_per_state_dim=6,
    bin_edges_per_action_dim=2,
    exploration_policy=RandomizedPolicy(generative_model),
    num_steps=int(1e5),
    multithreading=False,
)

print("Type of observation space: ", type(abstracted_model.observation_space))
print(
    f"The abstract observation space has {abstracted_model.observation_space.n} states!"
)
print("Type of environment: ", type(abstracted_model))

Simulation time: 0.9088s
Trajectories in dataset: 4499
Learning Abstraction: 2.2800s
Type of observation space:  <class 'gymnasium.spaces.discrete.Discrete'>
The abstract observation space has 1296 states!
Type of environment:  <class 'verigym.environments.explicitenv.ExplicitEnv'>


The abstract model allows us to explicitly access the model's transition and reward functions.

In [5]:
state = 100

print(f"Transitions from state {state}:")
pprint(abstracted_model.transition_function[state])
print()
print(f"State-action rewards of state {state}:")
pprint(abstracted_model.reward_function[state])

Transitions from state 100:
defaultdict(<function create_new_objects.<locals>.<lambda>.<locals>.<lambda> at 0x119faa340>, {})

State-action rewards of state 100:
defaultdict(<function create_new_objects.<locals>.<lambda>.<locals>.<lambda> at 0x119fa9ee0>, {})


## 3. Model checking

We now want to model check the abstraction we learned in `stormpy` w.r.t maximal cumulative reward. 

Therefore, we
1. Export the model to a `stormpy.storage.SparseMdp`
2. Model check the MDP in `stormpy` and receive a scheduler (policy)

In [6]:
stormpy_mdp = build_stormpy_mdp(abstracted_model)

print(stormpy_mdp)

-------------------------------------------------------------- 
Model type: 	MDP (sparse)
States: 	1296
Transitions: 	1468
Choices: 	1325
Reward Models:  reward0
State Labels: 	2 labels
   * deadlock -> 1265 item(s)
   * init -> 1 item(s)
Choice Labels: 	none
-------------------------------------------------------------- 



In [7]:
gamma = 0.95  # discount factor
prop = stormpy.parse_properties(f"Rmax=?[Cdiscount={gamma}]")[
    0
]  # checking for cumulative discounted reward

# Run the model checking:
result = stormpy.check_model_sparse(stormpy_mdp, prop, extract_scheduler=True)
scheduler = result.scheduler

## 4. Policy evaluation

Finally, we want to evaluate our model checked policy on the original and the abstract model.

1. We convert the `stormpy` policy to a `verigym.StormpyPolicy`
2. We apply it to both the original and the abstract model.

In [8]:
# policy mapped to the original model
verigym_policy = verigym.StormpyPolicy(scheduler, abstracted_model.abstraction_map)
# policy on the abstract model
verigym_policy_on_abstracted = verigym.StormpyPolicy(
    scheduler, abstraction_mapper=verigym.AbstractionMapper()
)

n_simulation_steps = int(10e4)

In [9]:
# simulation in the original (continuous) model
trajectories_original = generative_model.simulate(
    policy=verigym_policy,
    n_steps=n_simulation_steps,
)
mean_rewards_original = get_mean_reward_from_trajectories(trajectories_original)

Simulating:   0%|          | 0/100000 [00:00<?, ?it/s]

In [10]:
# verify the policy: (2) policy performance on abstracted model
trajectories_abstracted = abstracted_model.simulate(
    policy=verigym_policy_on_abstracted,
    n_steps=n_simulation_steps,
)
mean_rewards_abstracted = get_mean_reward_from_trajectories(trajectories_abstracted)

Simulating:   0%|          | 0/100000 [00:00<?, ?it/s]

In [11]:
print("Comparing the mean cumulative reward of the abstract policy:")
print("Original environment: ", mean_rewards_original)
print("Abstract environment: ", mean_rewards_abstracted)

Comparing the mean cumulative reward of the abstract policy:
Original environment:  68.63417982155113
Abstract environment:  13614.0


# Visualize abstract-policy on the original environment

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Roll out one episode of verigym_policy, capturing frames via gym's rgb_array rendering
render_env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, info = render_env.reset()
frames = [render_env.render()]

terminated = truncated = False
while not (terminated or truncated):
    # verigym_policy.get_action currently returns the action as a float
    # (e.g. array([0.])) rather than a plain int, so it is coerced here
    # to satisfy CartPole's Discrete(2) action space.
    action = int(round(float(np.asarray(verigym_policy.get_action(obs)).reshape(-1)[0])))
    obs, reward, terminated, truncated, info = render_env.step(action)
    frames.append(render_env.render())
render_env.close()

print(f"Episode length: {len(frames)} steps")

fig, ax = plt.subplots()
ax.axis("off")
img = ax.imshow(frames[0])


def _update(i):
    img.set_data(frames[i])
    return (img,)


anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=40, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())